# sgd-vanilla-from-scratch — ex1: single-step SGD: in-place update + zero the grad

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `sgd-vanilla-from-scratch`. Running the final beacon cell reports progress against the `Optimizer: SGD vanilla from scratch` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: SGD vanilla from scratch` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sgd-vanilla-from-scratch`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sgd-vanilla-from-scratch"
DD_SUBTOPIC = "Optimizer: SGD vanilla from scratch"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Vanilla SGD from scratch — quick refresher

The simplest possible optimizer. Given a list of parameters and a learning rate, one step of SGD is:

```
for p in params:
    p <- p - lr * p.grad
    p.grad <- None    # or zero — zero_grad-style
```

Two things to get right:

1. **In-place mutation of `p.array`.** `p.array -= lr * p.grad` is preferred over `p.array = p.array - lr * p.grad` because the same tensor object stays alive — any external reference (state dicts, checkpointers) keeps pointing to the live weights.
2. **Zero the grad after stepping.** Otherwise next backward call ACCUMULATES — you'd see two steps' worth of gradient on the next update. Setting `p.grad = None` is the cheapest reset; `p.grad.zero_()` also works.

No momentum, no weight decay, no Nesterov — that's all in `SGD`'s richer siblings. This is the 5-line baseline.

### Exercise 1 — single-step SGD: in-place update + zero the grad

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the vanilla-SGD update rule (`p -= lr * p.grad`) in place across a parameter list, then zero each parameter's grad so the next backward call doesn't double-accumulate.
> Keywords: sgd, optimizer, in-place-update, zero-grad, vanilla
> ```

**KCs targeted:** `sgd-vanilla-from-scratch`, `grad-accumulate-on-leaf`

Implement `sgd_step(params, lr)`. One pass of vanilla SGD over a list of `MiniTensor` parameters:

1. For each `p in params`:
   - Skip if `p.grad is None` (a parameter that didn't participate in the last forward — its grad slot is empty; updating with `None` would crash).
   - Otherwise: **in-place** update `p.array -= lr * p.grad`. Use `p.array -=` (or `.sub_`), NOT `p.array = p.array - ...` — the test asserts that the underlying tensor object is preserved.
2. After the update, set `p.grad = None` so the next backward call starts from a clean slate.

**Returns** `None`. Mutates `params` in place.

**Why in-place.** The same `p.array` tensor object stays alive — any external reference (state dicts, checkpointers, the `build_parents` dict you handed to the autograd dispatcher) keeps pointing to the live weights. Re-binding `p.array` to a new tensor would break those references.

**Why `p.grad = None` (and not `.zero_()`).** Both work, but `None` is cheaper (no memory write, just dropping the reference) and is what PyTorch recommends since 1.7. The next backward call checks `if p.grad is None: p.grad = first_contribution`, which is exactly the leaf-accumulate pattern from a prior atom.

No momentum, no weight decay, no Nesterov — just the 5-line baseline that ARENA's training-loop drill assembles.

In [ ]:
def sgd_step(params: list, lr: float) -> None:
    """One SGD step: p.array -= lr * p.grad in place, then p.grad = None."""
    raise NotImplementedError()


def _test_ex1():
    # --- update direction is correct: p -= lr * p.grad ---
    p1 = MiniTensor(t.tensor([1.0, 2.0, 3.0]), requires_grad=True)
    p1.grad = t.tensor([0.1, 0.2, 0.3])
    p2 = MiniTensor(t.tensor([10.0, 20.0]), requires_grad=True)
    p2.grad = t.tensor([1.0, 2.0])

    p1_array_id = id(p1.array)
    p2_array_id = id(p2.array)

    sgd_step([p1, p2], lr=0.1)

    # value check
    assert t.allclose(p1.array, t.tensor([0.99, 1.98, 2.97]), atol=1e-6), (
        f'p1 update wrong: {p1.array}'
    )
    assert t.allclose(p2.array, t.tensor([9.9, 19.8]), atol=1e-6), (
        f'p2 update wrong: {p2.array}'
    )

    # --- grad cleared after update ---
    assert p1.grad is None, 'p1.grad must be None after sgd_step'
    assert p2.grad is None, 'p2.grad must be None after sgd_step'

    # --- in-place update preserves the tensor object identity ---
    assert id(p1.array) == p1_array_id, (
        'p1.array must be the SAME tensor object after sgd_step — '
        'did you use `p.array = p.array - lr * p.grad`? Use `-=` or `.sub_`.'
    )
    assert id(p2.array) == p2_array_id, 'p2.array object identity broken'

    # --- learning rate scales: lr=0 produces no change ---
    p3 = MiniTensor(t.tensor([5.0, 5.0]), requires_grad=True)
    p3.grad = t.tensor([1.0, 1.0])
    before = p3.array.clone()
    sgd_step([p3], lr=0.0)
    assert t.allclose(p3.array, before), 'lr=0 must leave params unchanged'
    assert p3.grad is None, 'lr=0 must still clear the grad (zero-grad semantics)'

    # --- parameters with grad=None are SKIPPED (not crashed on) ---
    p4 = MiniTensor(t.tensor([7.0, 7.0]), requires_grad=True)
    p4.grad = None  # didn't participate in last forward
    p5 = MiniTensor(t.tensor([100.0]), requires_grad=True)
    p5.grad = t.tensor([10.0])
    before_p4 = p4.array.clone()
    sgd_step([p4, p5], lr=0.5)
    assert t.allclose(p4.array, before_p4), 'p4 had grad=None — must be left alone'
    assert t.allclose(p5.array, t.tensor([95.0])), 'p5 update wrong'
    assert p4.grad is None and p5.grad is None

    # --- empty params list is a no-op ---
    sgd_step([], lr=1.0)  # must not crash

    # --- two successive steps converge a tiny quadratic ---
    # Minimize (w - 5)^2; gradient is 2*(w - 5); start at w=0.
    w = MiniTensor(t.tensor([0.0]), requires_grad=True)
    for step in range(100):
        w.grad = 2 * (w.array - 5)
        sgd_step([w], lr=0.1)
    assert abs(w.array.item() - 5.0) < 1e-3, (
        f'convergence test failed: w should approach 5, got {w.array.item()}'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def sgd_step(params: list, lr: float) -> None:
    for p in params:
        if p.grad is None:
            continue
        # in-place: keep p.array as the SAME tensor object
        p.array -= lr * p.grad
        # zero the grad so the next backward starts fresh
        p.grad = None
```

**Why in-place mutation is non-negotiable.** A real training loop looks like:
```python
params = list(model.parameters())
optimizer = SGD(params, lr=0.01)
for batch in loader:
    loss = model(batch).backward()
    optimizer.step()
```
`params` was captured once. If `step()` re-binds `p.array = new`, the `params` list still points to the new arrays (via `p.array`), but anything else that held a reference to the OLD tensor object (state dict, checkpoint writer, an `nn.utils.parametrize` wrapper) is now stale. In-place `-=` keeps every external reference live.

**Why `p.grad = None` instead of `p.grad.zero_()`.** Both clear the grad, but `None` is faster (no tensor allocation persisted, no memory write), and signals to the next backward call that it should *create* a fresh grad tensor rather than overwriting. PyTorch's `optimizer.zero_grad(set_to_none=True)` is now the default since 2.0 for exactly this reason.

**Convergence sanity.** The included quadratic test `min (w-5)^2` is the minimal proof that the step rule is correct (direction AND scale). After 100 steps at lr=0.1, w should be within 1e-3 of 5 — the geometric decay rate is `|1 - 2*lr|^k = 0.8^100 ≈ 2e-10`, more than enough.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()